# 3B — Random Forest Regressor

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Author:** Ahmet Aybars Pektaş (91687)  

---

This notebook covers the **Random Forest Regressor** from **Work Package 3 — Model Implementation**.

**Deliverable:** D3.2 — Tuned Ensemble Models (Random Forest portion)

## 3B.1 — Imports & Load Raw Splits + Preprocessor

Load the **raw** train split and the **fitted preprocessing pipeline** that notebook 02 saved to `models/preprocessor_tree.pkl`. The pipeline is chained with the estimator below so cross-validation refits preprocessing on each fold (no target leakage from `MeanTargetEncoder`).

**Code organization note.** This notebook imports shared helpers from `src/utils.py` (`load_train`, `load_preprocessor`, `MODELS_DIR`, `RANDOM_STATE`). All four model notebooks (03a/03b/03c/04) use the same loaders so the train/test splits and preprocessor instances are byte-identical across the comparison — this prevents subtle apples-to-oranges bugs in §04's cross-model evaluation.

In [ ]:
# 3B.1 — Imports & Load Raw Splits + Preprocessor
import sys
import time

import joblib
import json
import numpy as np
import pandas as pd
from scipy.stats import randint
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    cross_validate,
    cross_val_predict,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

# src/ holds shared project utilities (utils.py) and the custom-transformer
# classes that joblib.load needs to resolve (preprocessing.py).
sys.path.insert(0, "../src")
import preprocessing  # noqa: F401  -- needed so joblib.load resolves custom transformers
from utils import (
    load_train,
    load_preprocessor,
    plot_pred_vs_actual,
    plot_residuals,
    plot_feature_importances,
    MODELS_DIR,
    RANDOM_STATE,
)

X_train, y_train = load_train()

# Tree-variant preprocessor: no scaling, no log transform on odometer.
# Trees are scale-invariant, so passthrough on numerics keeps year/odometer
# in their raw, interpretable units when reading split thresholds.
preprocessor_tree = load_preprocessor("tree")

# Pre-transform once for fast baseline experiments & CV.
# The full Pipeline(preprocessor + model) is assembled at the end (§3B.9)
# for the final .pkl export — that way the saved model can accept raw data.
X_train_processed = preprocessor_tree.transform(X_train)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Raw features ({X_train.shape[1]}): {list(X_train.columns)}")
print(f"X_train processed shape: {X_train_processed.shape}")
print(f"Loaded preprocessor: {type(preprocessor_tree).__name__} "
      f"with steps {[name for name, _ in preprocessor_tree.steps]}")

## 3B.2 — Model Setup

Initialize the `RandomForestRegressor` with sensible baseline defaults.
We cap `max_depth=20` to keep trees from growing enormous on 305K rows,
and set `n_jobs=-1` here (but `n_jobs=1` on CV later to avoid CPU oversubscription).

In [ ]:
# 3B.2 — Model Setup
rf_baseline = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print(f"Baseline model: {rf_baseline}")

## 3B.3 — Baseline Training (Default Hyperparameters)

Train the baseline Random Forest on pre-transformed data. Then run 5-fold CV.

**Performance note:** `n_jobs=-1` is on the model (parallel trees), so CV runs with `n_jobs=1`
to avoid oversubscription. Pre-transforming the data also eliminates repeated preprocessing per fold.

In [ ]:
# 3B.3 — Baseline Training (Default Hyperparameters)
print("Training baseline Random Forest on pre-transformed data...")
t0 = time.perf_counter()
rf_baseline.fit(X_train_processed, y_train)
baseline_fit_time = time.perf_counter() - t0
print(f"Training completed in {baseline_fit_time:.2f} seconds")

# n_jobs=1 on CV because the RF already uses all cores internally
print("\n5-fold cross-validation (default hyperparameters) ...")
cv_mae = cross_val_score(rf_baseline, X_train_processed, y_train, cv=5,
                         scoring="neg_mean_absolute_error", n_jobs=1)

baseline_score = -cv_mae.mean()
print(f"\nBaseline summary:")
print(f"  CV MAE = ${baseline_score:,.2f} ± ${cv_mae.std():,.2f}")

## 3B.4 — Hyperparameter Tuning

Use `RandomizedSearchCV` to tune key hyperparameters:
- `n_estimators`: 100–300
- `max_depth`: 15–25
- `min_samples_split`: 2, 5, 10, 20 (includes sklearn default of 2)
- `min_samples_leaf`: 1, 2, 5, 10 (includes sklearn default of 1)
- `max_features`: log2, 0.5, 0.7, None (None = all features, same as baseline default)

20 iterations × 3-fold CV = 60 fits. Estimated ~20–30 min on a local machine.

In [ ]:
# 3B.4 — Hyperparameter Tuning (RandomizedSearchCV)
param_distributions = {
    'n_estimators':      randint(100, 300),
    'max_depth':         randint(15, 25),
    'min_samples_split': [2, 5, 10, 20],       # 2 = sklearn default (no extra constraint)
    'min_samples_leaf':  [1, 2, 5, 10],        # 1 = sklearn default (no extra constraint)
    'max_features':      ['log2', 0.5, 0.7, None],  # None = all features
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=20,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=2,
    return_train_score=True,
)

print("Starting RandomizedSearchCV (20 iterations x 3 folds = 60 fits)...")
print("Estimated time: ~20-30 minutes on a local machine.\n")
t0 = time.perf_counter()
random_search.fit(X_train_processed, y_train)
search_time = time.perf_counter() - t0

print(f"\nSearch completed in {search_time/60:.1f} minutes")
print(f"\nBest parameters:")
for param, val in random_search.best_params_.items():
    print(f"  {param}: {val}")
print(f"\nBest CV MAE: ${-random_search.best_score_:,.2f}")

# Compare to the no-tuning reference from §3B.3
print(f"\nImprovement over baseline (default hyperparameters):")
print(f"  baseline CV MAE = ${baseline_score:,.2f}")
print(f"  tuned    CV MAE = ${-random_search.best_score_:,.2f}")
print(f"  delta           = ${baseline_score - (-random_search.best_score_):,.2f}")

## 3B.5 — Train Best Model

Retrain with the best hyperparameters found during tuning. Compare against the baseline
using 5-fold CV on equal footing, and automatically select whichever model performs better.

In [ ]:
# 3B.5 — Train Best Model
# Compare tuned vs baseline, pick the better one
best_params = random_search.best_params_

rf_tuned = RandomForestRegressor(
    **best_params,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Training tuned model with best hyperparameters...")
t0 = time.perf_counter()
rf_tuned.fit(X_train_processed, y_train)
train_time = time.perf_counter() - t0
print(f"Training completed in {train_time:.2f} seconds")

# 5-fold CV for both models on equal footing
print("\n5-fold CV on the tuned Random Forest ...")
res = cross_validate(
    rf_tuned, X_train_processed, y_train,
    cv=5,
    scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
    n_jobs=1,
    return_train_score=True,
)

tuned_mae       = -res["test_mae"]
tuned_r2        =  res["test_r2"]
tuned_train_mae = -res["train_mae"]   # used in §3B.7 overfit analysis

tuned_score = tuned_mae.mean()

print(f"\n  MAE per fold: " + ", ".join(f"${m:>7,.0f}" for m in tuned_mae))
print(f"  R^2 per fold: " + ", ".join(f" {r:.4f}" for r in tuned_r2))
print(f"\nTuned Random Forest:")
print(f"  CV MAE = ${tuned_score:,.2f} ± ${tuned_mae.std():,.2f}")
print(f"  CV R^2 = {tuned_r2.mean():.4f} ± {tuned_r2.std():.4f}")
print(f"\nCV MAE (baseline): ${baseline_score:,.2f}")

# Pick the winner
if tuned_score < baseline_score:
    rf_best = rf_tuned
    print(f"\nTuned model wins by ${baseline_score - tuned_score:,.2f} MAE. Using tuned model.")
else:
    rf_best = rf_baseline
    print(f"\nBaseline model is equal or better. Using baseline model.")
    print("(This can happen when default params are already near-optimal for this dataset.)")

## 3B.6 — Diagnostic Plots (Predicted vs. Actual + Residuals)

Two CV-based diagnostic plots on the best Random Forest variant:

- **Predicted vs. Actual** — does the model systematically over- or under-predict in some price ranges?
- **Residual distribution** — is the error roughly symmetric around zero, or skewed?

We use `cross_val_predict` so each prediction is made by a model that *did not* see that row during training. This keeps the plots honest without touching the held-out test set (which is reserved for §04).

The plotting helpers live in `src/utils.py` and are reused by 03a, 03c, and 04 to keep visual comparisons consistent across notebooks.

In [ ]:
# 3B.6 — Diagnostic Plots (Predicted vs. Actual + Residuals)
# Out-of-fold predictions (each row predicted by a model that didn't see it).
print("Generating out-of-fold predictions for diagnostic plots ...")
t0 = time.perf_counter()
y_pred_cv = cross_val_predict(rf_best, X_train_processed, y_train, cv=5, n_jobs=1)
print(f"  done in {(time.perf_counter() - t0)/60:.1f} min")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_pred_vs_actual(y_train, y_pred_cv, ax=axes[0],
                    title="Random Forest (tuned): predicted vs actual (CV)")
plot_residuals(y_train, y_pred_cv, ax=axes[1],
               title="Random Forest (tuned): residual distribution (CV)")
# Prices and predicted prices are >= 0 -- pin scatter axes to start at 0.
axes[0].set_xlim(left=0)
axes[0].set_ylim(bottom=0)
# Constrain residual view to +/- 50k for readability.
axes[1].set_xlim(-50_000, 50_000)
plt.tight_layout()
plt.show()

## 3B.7 — Feature Importance

Extract and visualize feature importances from the Random Forest (MDI — Mean Decrease in Impurity).
Feature names are taken directly from the preprocessed DataFrame columns
(preserved by `set_output(transform="pandas")` in the preprocessing pipeline).

The plotting helper from `src/utils.py` keeps the chart style consistent across 03a, 03c, and 04.

In [ ]:
# 3B.7 — Feature Importance
feature_names = X_train_processed.columns.tolist()
importances = rf_best.feature_importances_

# Top 15 table
top_idx = np.argsort(importances)[::-1][:15]

# Clean up column names for display (strip transformer prefix)
display_names = [n.split('__', 1)[-1] if '__' in n else n for n in feature_names]

print("Top 15 features by Random Forest MDI importance:")
print(pd.DataFrame({
    "feature":    [display_names[i] for i in top_idx],
    "importance": importances[top_idx],
}).to_string(index=False))

# Plot via utils for cross-notebook consistency
fig, ax = plt.subplots(figsize=(10, 6))
plot_feature_importances(importances, feature_names, n=15, ax=ax,
                         title="Random Forest — top 15 features by MDI importance")
plt.tight_layout()
plt.show()

## 3B.8 — Overfitting Analysis

The train-vs-CV gap is a direct overfitting signal: if the model fits training data dramatically better than held-out folds, it has memorized noise. We compute the gap from §3B.5's `return_train_score=True` CV pass — same folds, same hyperparameters, both train and validation scores already in hand.

Rule of thumb for tabular regression on a dataset of this size: a gap below ~10 % of the mean MAE is comfortable, 10–25 % is acceptable, anything larger suggests the model needs more regularization.

In [ ]:
# 3B.8 — Overfitting Analysis
gap     = tuned_train_mae.mean() - tuned_mae.mean()   # CV MAE > train MAE
gap_pct = 100 * gap / tuned_mae.mean()

print(f"Train MAE (CV folds, mean):  ${tuned_train_mae.mean():,.2f}")
print(f"CV    MAE (CV folds, mean):  ${tuned_mae.mean():,.2f}")
print(f"Gap (CV - train):            ${gap:,.2f}  ({gap_pct:+.1f}% of CV MAE)")

if abs(gap_pct) < 10:
    verdict = "comfortable -- model generalizes well"
elif abs(gap_pct) < 25:
    verdict = "acceptable -- mild overfit, typical for tree ensembles"
else:
    verdict = "concerning -- consider tighter regularization (lower max_depth, higher min_samples_leaf)"
print(f"\nVerdict: {verdict}")

## 3B.9 — Save Model

Compose the fitted preprocessor and the best Random Forest into a single `Pipeline`,
then serialize it to `models/random_forest.pkl`. This way, Notebook 04 can call
`.predict()` on raw (unprocessed) data directly.

Also write `models/rf_tuning_results.json` with the search summary — same structure
as the XGBoost sidecar JSON so §04 can present them side by side.

In [ ]:
# 3B.9 — Save Model
full_pipeline = Pipeline([
    ("preprocess", preprocessor_tree),
    ("model", rf_best),
])

# Sanity check: predict on raw X_train (not pre-transformed)
y_pred_check = full_pipeline.predict(X_train)
check_mae = float(np.mean(np.abs(y_train - y_pred_check)))
train_direct_mae = float(np.mean(np.abs(y_train - rf_best.predict(X_train_processed))))
print(f"Full pipeline MAE on raw train data: ${check_mae:,.2f}")
print(f"Direct model MAE (pre-transformed):  ${train_direct_mae:,.2f}")
assert np.isclose(check_mae, train_direct_mae, rtol=1e-5), \
    'Pipeline and direct predictions should match!'
print("Pipeline predictions match direct predictions\n")

# Canonical path -- §04 (evaluation) loads exactly this filename.
save_path = MODELS_DIR / "random_forest.pkl"
joblib.dump(full_pipeline, save_path)
# Print project-relative path so cell output is portable across machines
print(f"Saved -> models/{save_path.name}")

In [ ]:
# Save tuning results JSON
tuning_results = {
    "best_params": random_search.best_params_,
    "best_cv_mae": -random_search.best_score_,
    "baseline_cv_mae": baseline_score,
    "param_distributions": {
        k: v.args if hasattr(v, 'args') else v   # unwrap scipy randint
        for k, v in param_distributions.items()
    },
    "cv_results_summary": [
        {
            "rank": int(random_search.cv_results_["rank_test_score"][i]),
            "params": random_search.cv_results_["params"][i],
            "mean_test_mae": float(-random_search.cv_results_["mean_test_score"][i]),
            "mean_train_mae": float(-random_search.cv_results_["mean_train_score"][i]),
        }
        for i in range(len(random_search.cv_results_["params"]))
    ],
    "n_iter": random_search.n_iter,
    "cv_folds": random_search.cv,
}

# Convert numpy types to plain Python so JSON doesn't choke
def make_serializable(obj):
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [make_serializable(x) for x in obj]
    if hasattr(obj, 'item'):  # numpy scalar
        return obj.item()
    return obj

tuning_results = make_serializable(tuning_results)

with open(MODELS_DIR / "rf_tuning_results.json", "w") as f:
    json.dump(tuning_results, f, indent=2)

print(f"Saved -> models/rf_tuning_results.json")
print(f"Best params: {tuning_results['best_params']}")
print(f"Best CV MAE: ${tuning_results['best_cv_mae']:,.2f}")

## 3B.10 — Summary

**Tuned Random Forest metrics (5-fold CV on training set — see §3B.5):**
- The tuned model improves over the baseline by selecting optimal `max_depth`, `max_features`, `min_samples_leaf`, and `min_samples_split` via RandomizedSearchCV with MAE as the selection criterion.
- R² and MAE both substantially outperform the linear baseline from 03a, as expected — trees capture non-linear depreciation and interaction effects that a linear model cannot represent.

**Overfitting note:** The train-vs-CV MAE gap (§3B.8) should be monitored. Random Forests with deep trees tend to overfit on tabular data; if the gap is large, consider capping `max_depth` or raising `min_samples_leaf`.

**What happens next:**
- The final cross-model comparison happens in `04_evaluation.ipynb`, which loads `linear_regression.pkl`, `random_forest.pkl`, and `xgboost.pkl` and evaluates each on the held-out test set.